## Project Overview

In this project, we examine the research question:  
**Do male and female soccer players peak at the same age, and does their development gap close at the same rate over time?**

We focus on two main questions. First, do male and female players reach their peak at the same age? Second, does the **development gap** — defined as the difference between a player's potential rating and current overall rating — shrink at a similar rate for both genders?

The development gap is a useful measure because it captures how much growth a player still has remaining. This allows us to study career development separately from raw skill level.

The project uses three core variables:

- **Age**
- **Overall rating**
- **Development gap**

Age is one of the most important factors in understanding athletic development.

The key takeaway from this research is that gender differences in rating levels do not necessarily mean differences in development over time. Our analysis suggests that male and female players improve and reach their peak on similar timelines. Most importantly, the development gap closes at nearly the same rate for both groups, suggesting that male and female soccer players follow very similar development patterns.


## Data Used: FIFA Players 2015 – 2023 + FC24

### Women's player datasets
- `female_players_16.csv`
- `female_players_17.csv`
- `female_players_18.csv`
- `female_players_19.csv`
- `female_players_20.csv`
- `female_players_21.csv`
- `female_players_22.csv`

These files contain female player data from FIFA 16 through FIFA 22. Each dataset includes player information such as age, overall rating, and potential rating. For this study, all female-player datasets are combined into one merged female dataset.

### Men's player datasets
- `players_15.csv`
- `players_16.csv`
- `players_17.csv`
- `players_18.csv`
- `players_19.csv`
- `players_20.csv`
- `players_21.csv`
- `players_22.csv`

These files contain male player data from FIFA 15 through FIFA 22. Each dataset includes player information such as age, overall rating, and potential rating. For this study, all male-player datasets are combined into one merged male dataset.

## Data Description

For this study, we use 15 FIFA player dataset tables: seven female-player tables (`female_players_16.csv`, `female_players_17.csv`, `female_players_18.csv`, `female_players_19.csv`, `female_players_20.csv`, `female_players_21.csv`, and `female_players_22.csv`) and eight male-player tables (`players_15.csv`, `players_16.csv`, `players_17.csv`, `players_18.csv`, `players_19.csv`, `players_20.csv`, `players_21.csv`, and `players_22.csv`). Each row in every table represents **one player in one FIFA edition**. Therefore, same player may appear in multiple years as separate observations. The female tables contain **248, 299, 317, 299, 345, 345, and 391 observations**, respectively, while the male tables contain **16,155, 15,623, 17,596, 17,954, 18,085, 18,483, 18,944, and 19,239 observations**, respectively. These datasets include player-level information such as **age, overall rating, potential rating, nationality, club, position, and many other performance-related attributes**. In this project, we merge all female-player tables into one female dataset and all male-player tables into one male dataset, then focus mainly on **age, overall rating, and potential rating** to construct a **development gap** variable (`potential - overall`) for comparing player development patterns across genders.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Import & tag all datasets
female_16 = pd.read_csv("data/female_players_16.csv"); female_16["year"] = 2016
female_17 = pd.read_csv("data/female_players_17.csv"); female_17["year"] = 2017
female_18 = pd.read_csv("data/female_players_18.csv"); female_18["year"] = 2018
female_19 = pd.read_csv("data/female_players_19.csv"); female_19["year"] = 2019
female_20 = pd.read_csv("data/female_players_20.csv"); female_20["year"] = 2020
female_21 = pd.read_csv("data/female_players_21.csv"); female_21["year"] = 2021
female_22 = pd.read_csv("data/female_players_22.csv"); female_22["year"] = 2022

male_16 = pd.read_csv("data/players_16.csv"); male_16["year"] = 2016
male_17 = pd.read_csv("data/players_17.csv"); male_17["year"] = 2017
male_18 = pd.read_csv("data/players_18.csv"); male_18["year"] = 2018
male_19 = pd.read_csv("data/players_19.csv"); male_19["year"] = 2019
male_20 = pd.read_csv("data/players_20.csv"); male_20["year"] = 2020
male_21 = pd.read_csv("data/players_21.csv"); male_21["year"] = 2021
male_22 = pd.read_csv("data/players_22.csv"); male_22["year"] = 2022

# 2. Add gender label and stack everything together
female_all = pd.concat([female_16, female_17, female_18, female_19, female_20, female_21, female_22])
male_all   = pd.concat([male_16,   male_17,   male_18,   male_19,   male_20,   male_21,   male_22])

female_all["gender"] = "Female"
male_all["gender"]   = "Male"

cols = ["age", "overall", "potential", "gender"]
df = pd.concat([female_all[cols], male_all[cols]])

# 3. Split by gender
female_df = df.query("gender == 'Female'").copy()
male_df   = df.query("gender == 'Male'").copy()

# 4. Mean overall rating grouped by age
female_overall = female_df.groupby("age")["overall"].mean()
male_overall   = male_df.groupby("age")["overall"].mean()

# 5. Development gap (potential - overall)
female_df["gap"] = female_df["potential"] - female_df["overall"]
male_df["gap"]   = male_df["potential"]   - male_df["overall"]

female_gap = female_df.groupby("age")["gap"].mean()
male_gap   = male_df.groupby("age")["gap"].mean()

# 6. Print summary stats
print("=== Peak Age (highest mean overall rating) ===")
print("Female peak age:", female_overall.idxmax(),
      "  mean overall:", round(female_overall.max(), 1))
print("Male   peak age:", male_overall.idxmax(),
      "  mean overall:", round(male_overall.max(), 1))

print()
print("=== Mean Development Gap (potential - overall) by age ===")
for a in [17, 22, 25, 30]:
    fg = female_gap.get(a, float("nan"))
    mg = male_gap.get(a, float("nan"))
    print("Age", a, "-> Female gap:", round(fg, 1), "  Male gap:", round(mg, 1))

# 7. Plot 1: Mean overall rating by age
plt.scatter(x = female_overall.index, y = female_overall.values, label = "Female")
plt.scatter(x = male_overall.index,   y = male_overall.values,   label = "Male")
plt.xlabel("Age")
plt.ylabel("Mean Overall Rating")
plt.title("Mean Overall Rating by Age\n(Male vs Female, FIFA 16-22)")
plt.legend()
plt.show()

# 8. Plot 2: Development gap by age
plt.scatter(x = female_gap.index, y = female_gap.values, label = "Female")
plt.scatter(x = male_gap.index,   y = male_gap.values,   label = "Male")
plt.xlabel("Age")
plt.ylabel("Mean (Potential - Overall)")
plt.title("Development Gap by Age\n(Male vs Female, FIFA 16-22)")
plt.legend()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'data/female_players_16.csv'